# Étape 4 — Phase 0 : préparer Junyi 2015 pour l'entraînement

Ce notebook transforme les réponses brutes de Junyi en données prêtes pour les modèles de knowledge tracing (KT) :

1. **Graphe expert G\*** : les flèches de prérequis des experts, vérifiées (pas de cycle, exercices connus).
2. **Échantillon d'élèves** : élèves ayant au moins 10 réponses, puis un échantillon aléatoire (20 000 par défaut).
3. **Séquences** : pour chaque élève, ses réponses dans l'ordre du temps, découpées en morceaux de 200 au plus.
4. **Découpage par élève** : 70 % entraînement, 10 % validation, 20 % test.
5. **Diagnostics** pour l'article, dont le biais « ordre du cours » signalé dans la revue de littérature.

Pas besoin de GPU. Durée : 10 à 20 minutes. Tout est enregistré dans `projet_kt/junyi_prepare/`.

In [ ]:
# Cellule 1 — Paramètres (vous pouvez les modifier ici, et seulement ici)
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd

DOSSIER  = '/content/drive/MyDrive/projet_kt'
JUNYI    = os.path.join(DOSSIER, 'junyi2015')
SORTIE   = os.path.join(DOSSIER, 'junyi_prepare')
os.makedirs(SORTIE, exist_ok=True)

N_ELEVES   = 20_000   # taille de l'échantillon d'élèves (None = tous les élèves)
MIN_INTER  = 10       # nombre minimal de réponses par élève
LONG_MAX   = 200      # longueur maximale d'une séquence (les plus longues sont découpées)
PARTS      = {'train': 0.7, 'valid': 0.1, 'test': 0.2}
GRAINE     = 42

CONFIG = dict(N_ELEVES=N_ELEVES, MIN_INTER=MIN_INTER, LONG_MAX=LONG_MAX, PARTS=PARTS, GRAINE=GRAINE)
print(CONFIG)

In [ ]:
# Cellule 2 — Graphe expert G* (à partir de la colonne « prerequisites »)
!pip install -q networkx
import networkx as nx

exo = pd.read_csv(os.path.join(JUNYI, 'junyi_Exercise_table.csv'))
exo = exo.drop_duplicates(subset='name').reset_index(drop=True)
exo['exo_id'] = np.arange(len(exo))            # identifiant numérique 0..N-1, utilisé par les modèles
nom_vers_id = dict(zip(exo['name'], exo['exo_id']))

aretes, inconnues = [], 0
for _, r in exo.iterrows():
    if isinstance(r['prerequisites'], str):
        for p in r['prerequisites'].split(','):
            p = p.strip()
            if not p:
                continue
            if p in nom_vers_id:
                aretes.append((nom_vers_id[p], r['exo_id']))
            else:
                inconnues += 1
aretes = sorted(set(a for a in aretes if a[0] != a[1]))

G = nx.DiGraph()
G.add_nodes_from(exo['exo_id'])
G.add_edges_from(aretes)
import itertools
cycles = list(itertools.islice(nx.simple_cycles(G), 10_000)) if not nx.is_directed_acyclic_graph(G) else []  # plafonné

# Si des cycles existent, on retire le minimum d'arêtes pour obtenir un graphe acyclique (et on le note)
retirees = []
while not nx.is_directed_acyclic_graph(G):
    cyc = nx.find_cycle(G)
    retirees.append(cyc[-1][:2])
    G.remove_edge(*cyc[-1][:2])

stats_graphe = {
    'exercices': len(exo),
    'aretes_expert': G.number_of_edges(),
    'prerequis_vers_exercice_inconnu_ignores': inconnues,
    'cycles_detectes': len(cycles),
    'aretes_retirees_pour_acyclicite': len(retirees),
    'exercices_sans_prerequis': int(sum(1 for n in G if G.in_degree(n) == 0)),
    'exercices_isoles': int(sum(1 for n in G if G.degree(n) == 0)),
    'profondeur_max': int(nx.dag_longest_path_length(G)),
}
pd.DataFrame(sorted(G.edges()), columns=['A', 'B']).to_csv(os.path.join(SORTIE, 'graphe_expert_aretes.csv'), index=False)
exo[['exo_id', 'name', 'topic', 'area']].to_csv(os.path.join(SORTIE, 'exercices.csv'), index=False)
A_expert = nx.to_numpy_array(G, nodelist=list(exo['exo_id']), dtype=np.int8)
np.save(os.path.join(SORTIE, 'adjacence_expert.npy'), A_expert)
for k, v in stats_graphe.items():
    print(f'{k:42s} {v}')

In [ ]:
# Cellule 3 — Colonnes du journal et premier passage : nombre de réponses par élève
CHEMIN_LOG = os.path.join(JUNYI, 'junyi_ProblemLog_original.csv')
entete = pd.read_csv(CHEMIN_LOG, nrows=0).columns.tolist()
def trouver(*candidats):
    for c in candidats:
        if c in entete:
            return c
    for c in entete:
        if any(x in c for x in candidats):
            return c
    raise KeyError(f'Colonne introuvable parmi {candidats} ; colonnes : {entete}')
COL_U, COL_E = trouver('user_id', 'user'), trouver('exercise')
COL_C, COL_T = trouver('correct'), trouver('time_done', 'timestamp', 'time')
print('Colonnes utilisées :', COL_U, COL_E, COL_C, COL_T)

t0 = time.time()
nb_par_eleve = pd.read_csv(CHEMIN_LOG, usecols=[COL_U])[COL_U].value_counts()
eligibles = nb_par_eleve[nb_par_eleve >= MIN_INTER].index
rng = np.random.default_rng(GRAINE)
if N_ELEVES and N_ELEVES < len(eligibles):
    choisis = set(rng.choice(np.asarray(eligibles), size=N_ELEVES, replace=False).tolist())
else:
    choisis = set(eligibles.tolist())
print(f'Élèves : {len(nb_par_eleve):,} au total, {len(eligibles):,} avec ≥ {MIN_INTER} réponses, '
      f'{len(choisis):,} retenus ({(time.time()-t0)/60:.1f} min)')

In [ ]:
# Cellule 4 — Second passage : réponses des élèves retenus, triées dans le temps
t0 = time.time()
morceaux = []
for m in pd.read_csv(CHEMIN_LOG, usecols=[COL_U, COL_E, COL_C, COL_T], chunksize=2_000_000):
    morceaux.append(m[m[COL_U].isin(choisis)])
log = pd.concat(morceaux, ignore_index=True)
del morceaux

log['exo_id'] = log[COL_E].map(nom_vers_id)
hors_table = int(log['exo_id'].isna().sum())
log = log.dropna(subset=['exo_id'])
log['exo_id'] = log['exo_id'].astype(int)
log['r'] = log[COL_C].astype(str).str.strip().str.lower().isin(['true', '1', '1.0']).astype(np.int8)
if not np.issubdtype(log[COL_T].dtype, np.number):
    log[COL_T] = pd.to_datetime(log[COL_T], errors='coerce').astype('int64')
log = log.sort_values([COL_U, COL_T], kind='stable').reset_index(drop=True)
print(f'{len(log):,} réponses retenues | {hors_table:,} ignorées (exercice absent de la table) | '
      f'{(time.time()-t0)/60:.1f} min')

In [ ]:
# Cellule 5 — Découpage PAR ÉLÈVE (70/10/20) et construction des séquences
eleves = np.array(sorted(log[COL_U].unique()))
rng = np.random.default_rng(GRAINE)
rng.shuffle(eleves)
n = len(eleves)
coupe1, coupe2 = int(PARTS['train'] * n), int((PARTS['train'] + PARTS['valid']) * n)
partie_de = {u: 'train' for u in eleves[:coupe1]}
partie_de.update({u: 'valid' for u in eleves[coupe1:coupe2]})
partie_de.update({u: 'test' for u in eleves[coupe2:]})

lignes = []
for u, g in log.groupby(COL_U, sort=False):
    ex, rep, ts = g['exo_id'].tolist(), g['r'].tolist(), g[COL_T].tolist()
    for debut in range(0, len(ex), LONG_MAX):
        fin = debut + LONG_MAX
        if len(ex[debut:fin]) < 2:          # un morceau d'une seule réponse n'apprend rien
            continue
        lignes.append({'uid': u, 'partie': partie_de[u], 'morceau': debut // LONG_MAX,
                       'exercices': ','.join(map(str, ex[debut:fin])),
                       'reponses': ','.join(map(str, rep[debut:fin])),
                       'temps': ','.join(map(str, ts[debut:fin]))})
seqs = pd.DataFrame(lignes)
for p in PARTS:
    seqs[seqs.partie == p].to_csv(os.path.join(SORTIE, f'sequences_{p}.csv'), index=False)
print(seqs.groupby('partie').agg(eleves=('uid', 'nunique'), sequences=('uid', 'size')))

In [ ]:
# Cellule 6 — Statistiques pour le tableau « Datasets » de l'article
long_seq = seqs['exercices'].str.count(',') + 1
exos_train = set(log[log[COL_U].map(partie_de) == 'train']['exo_id'])
aretes_couvertes = sum(1 for a, b in G.edges() if a in exos_train and b in exos_train)

stats = {
    'eleves': int(n),
    'reponses': int(len(log)),
    'exercices_vus': int(log['exo_id'].nunique()),
    'taux_reussite': round(float(log['r'].mean()), 3),
    'reponses_par_eleve_mediane': float(log.groupby(COL_U).size().median()),
    'sequences': int(len(seqs)),
    'longueur_sequence_moyenne': round(float(long_seq.mean()), 1),
    'aretes_expert': int(G.number_of_edges()),
    'aretes_dont_les_2_exercices_sont_vus_en_train': int(aretes_couvertes),
}
for k, v in stats.items():
    print(f'{k:48s} {v:,}' if isinstance(v, int) else f'{k:48s} {v}')

### Diagnostic « ordre du cours »

Pour chaque flèche experte A → B, parmi les élèves qui ont travaillé les deux exercices, on regarde **dans quel ordre** ils les ont faits pour la première fois.

- Si presque tout le monde fait A avant B, la plateforme impose sans doute cet ordre. Les réponses des élèves ne permettront alors pas de distinguer un vrai prérequis d'un simple ordre de présentation (le risque soulevé par l'article PKT).
- Si l'ordre varie, les données comportementales ont un vrai pouvoir de vérification.

Ce chiffre prépare directement la discussion de RQ4.

In [ ]:
# Cellule 7 — Diagnostic « ordre du cours » (sur l'échantillon d'entraînement)
train = log[log[COL_U].map(partie_de) == 'train']
premier = train.groupby([COL_U, 'exo_id'])[COL_T].min().unstack()   # élèves × exercices : date du 1er essai

lignes = []
for a, b in G.edges():
    if a in premier.columns and b in premier.columns:
        ta, tb = premier[a], premier[b]
        les_deux = ta.notna() & tb.notna()
        k = int(les_deux.sum())
        if k >= 20:
            lignes.append({'A': a, 'B': b, 'eleves_les_deux': k,
                           'part_A_avant_B': float((ta[les_deux] < tb[les_deux]).mean())})
ordre = pd.DataFrame(lignes)
ordre.to_csv(os.path.join(SORTIE, 'diagnostic_ordre.csv'), index=False)

if len(ordre):
    stats['aretes_avec_20_eleves_ou_plus'] = int(len(ordre))
    stats['part_A_avant_B_mediane'] = round(float(ordre.part_A_avant_B.median()), 3)
    stats['aretes_ordre_quasi_impose_(>=95%)'] = int((ordre.part_A_avant_B >= 0.95).sum())
    stats['aretes_ordre_variable_(<80%)'] = int((ordre.part_A_avant_B < 0.80).sum())
    print(ordre.part_A_avant_B.describe().round(3))
    for k in ['aretes_avec_20_eleves_ou_plus', 'part_A_avant_B_mediane',
              'aretes_ordre_quasi_impose_(>=95%)', 'aretes_ordre_variable_(<80%)']:
        print(f'{k:40s} {stats[k]}')
else:
    print("Pas assez d'élèves par arête pour ce diagnostic.")

In [ ]:
# Cellule 8 — Sauvegarde et résumé à copier dans la conversation
resume = {'config': CONFIG, 'graphe': stats_graphe, 'donnees': stats}
with open(os.path.join(SORTIE, 'resume_phase0.json'), 'w', encoding='utf-8') as f:
    json.dump(resume, f, ensure_ascii=False, indent=2, default=int)
print(json.dumps(resume, ensure_ascii=False, indent=2, default=int))
print('\nFichiers créés :', sorted(os.listdir(SORTIE)))